# 12.9 內建二分搜尋模組：bisect 與數值區間查詢應用

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_12-9_bisect_module_and_range_queries.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**先備知識**：已掌握 Chapter 12.7 精確二分搜尋模型與 Chapter 12.8 邊界二分搜尋（Lower / Upper Bound）。

---

### 學習導覽：官方神兵降臨——標準庫 bisect 與現代演算法實戰

恭喜你抵達了第十二章「排序與搜尋演算法」的終極殿堂！
在上一小節（12.8 節）中，我們經歷了扎實的演算法底層修煉，親手手刻出了 Lower Bound（第一個 $\ge x$）與 Upper Bound（第一個 $> x$）。你已經深刻理解了雙邊界的數學本質與防死結心法。

現在，我們要迎來 Python 官方為全球演算法選手準備的無上神兵——**標準函式庫 `bisect` 模組**！
為什麼 Python 官方要專門打造一個 `bisect` 模組？
- **純 C 語言極速編譯**：底層由高度優化的 C 語言打造，執行速度比純手刻 Python 迴圈快上數倍！
- **完美對映下界與上界**：
  - `bisect.bisect_left(a, x)` 正是我們剛剛手刻的 **Lower Bound**！
  - `bisect.bisect_right(a, x)` 正是我們剛剛手刻的 **Upper Bound**！
- **消滅義大利麵條條件句**：面對複雜的階梯分級（如 60 分及格、70 分丙等、80 分乙等、90 分甲等），利用 `bisect` 查表法，只要 1 行程式碼就能秒殺原本需要寫 10 個 `if-elif` 的繁瑣分支！
- **動態維護有序序列**：透過 `bisect.insort()`，新數字隨時插入，序列永遠自動保持升序！

在本單元中，我們將透過 6 個平緩的微階梯，完成第十二章的終極昇華：
1. **12.9.1 標準函式庫 `import bisect`**：底層 C 語言優化優勢與引入規範。
2. **12.9.2 `bisect.bisect_left(a, x)`**：精準對應 Lower Bound（第一個 $\ge x$ 插入位置）。
3. **12.9.3 `bisect.bisect_right(a, x)`**：精準對應 Upper Bound（第一個 $> x$ 插入位置）。
4. **12.9.4 數值區間計數神技**：`bisect_right - bisect_left` 秒求出現次數與範圍計數。
5. **12.9.5 數值級距查表神器**：利用 bisect 快速實作等第成績評定（免寫冗長 if-elif）。
6. **12.9.6 維護動態有序序列**：`bisect.insort()` 高效插入與單調答案值域二分搜尋初探。

讓我們拔出這把官方瑞士軍刀，瀟灑征戰所有競賽考場！

### 12.9.1 標準函式庫 `import bisect`：底層 C 語言優化優勢與引入規範

#### 1. 生活故事比喻：工匠純手工打造 vs 國家級航太自動化工廠
在 12.7 和 12.8 節中，我們像一位勤勞的古代工匠，手拿鐵鎚一錘一錘敲出了精緻的二分搜尋齒輪與指針。這段手刻的過程無比珍貴，因為它讓你徹底看清了齒輪咬合的每一個精密細節。
然而，在分秒必爭的 APCS 考場與工業級軟體系統中，如果每次要用二分搜尋都要重新敲 20 行齒輪，不僅浪費寶貴時間，更有可能因為一聲咳嗽粗心打錯一個等號。
Python 官方早已在標準庫中設立了一座「國家級航太自動化工廠」——**`bisect` 模組**！
它直接內建在 Python 直譯器中（無需 `pip install` 安裝），由世界頂級軟體大師用純 C 語言編寫與優化，速度快如閃電，拿來即可直接投入實戰！

#### 2. 底層運作機制：C 語言加速與單詞詞源解密
- **詞源解析**：`bisect` 這個單字源自拉丁文與幾何學，意為**「一分為二、二等分（Bisection）」**，完美揭示了其折半搜尋的核心本質。
- **引入規範**：標準引入語法為 `import bisect`。
- **底層優化**：在 CPython 直譯器中，`bisect` 模組底層有對應的 C 實作擴展（`_bisect`）。當它在百萬筆級別的資料上運作時，指令直接在 CPU 暫存器層級高速流轉，比純 Python 直譯的 `while` 迴圈更省記憶體、速度更快！

#### 3. 初學者常見陷阱：以為 bisect 會自動幫未排序數列排序
請牢記第 12.7.1 節的鐵律：
`bisect` 是二分搜尋工具，**它預設傳入的串列「已經是排序好的」！**
如果傳入一個未排序的雜亂串列給 `bisect`，它**絕對不會幫你自動排序**，而是會盲目在混亂資料中折半，回傳毫無意義的垃圾位置！

#### 4. APCS 實戰視野
在 APCS 實作測驗中，標準函式庫 `bisect` 是 100% 官方允許且大力推薦使用的合法模組。熟練調用 `bisect`，能讓你在考場上省下整整 15 分鐘的手刻與除錯時間，堪稱競賽選手的合法作弊神器！

In [ ]:
# 範例 12.9.1：引入 bisect 模組與基本功能檢視
import bisect

# 檢視 bisect 模組提供的核心工具清單
tools = [item for item in dir(bisect) if not item.startswith("_")]
print("bisect 模組核心法寶清單：", tools)

# 宣告一組已排序的數列
sorted_arr = [10, 20, 30, 40, 50]
print("\n已排序數列：", sorted_arr)

# 測試最基礎的定位查詢
# bisect.bisect(a, x) 預設等價於 bisect_right
pos = bisect.bisect(sorted_arr, 25)
print(f"數值 25 應該插入在索引 {pos}，插入後依然維持升序！")
print(f"驗證：前一位是 {sorted_arr[pos-1]}，後一位是 {sorted_arr[pos]}")

In [ ]:
# 填空題 12.9.1：引入與呼叫標準庫
# 任務：正確引入 bisect 模組並查詢數值 35 應插入的位置。
___ bisect

primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37]

# 呼叫 bisect 模組的查詢功能
insert_pos = bisect.___(primes, 35)

print("質數清單：", primes)
print("數值 35 的合適插入索引為：", insert_pos)
print(f"插入位置夾在 {primes[insert_pos-1]} 與 {primes[insert_pos]} 之間！")

In [ ]:
# ==========================================
# [4] Code 練習題 12.9.1
# 任務說明：
# 給定一組未排序的整數清單 raw_data 與搜尋目標 target。
# 請撰寫程式：
# 1. 確保 raw_data 進行升序排序。
# 2. 引入 bisect 模組，使用 bisect.bisect(raw_data, target) 找出合適位置。
# 3. 印出排序後的清單與該位置。
#
# 【公開測試資料 1】
# raw_data = [50, 10, 40, 20, 30]
# target = 25
# 預期輸出：
# 排序後串列： [10, 20, 30, 40, 50]
# 插入位置索引： 2
#
# 【公開測試資料 2】
# raw_data = [99, 1]
# target = 50
# 預期輸出：
# 排序後串列： [1, 99]
# 插入位置索引： 1
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
import bisect

raw_data = [50, 10, 40, 20, 30]
target = 25

raw_data.sort()
print("排序後串列：", raw_data)
pos = bisect.bisect(raw_data, target)
print("插入位置索引：", pos)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.9.1
# 任務說明：
# 某量測紀錄如下：readings = [12.5, 3.8, 45.2, 19.0, 28.4]
# 請寫出程式：先對其排序，接著使用 bisect 找出數值 0.0 與 100.0 各自會被安排在什麼索引位置，
# 驗證兩極端數值分別會停在 0 與 len(readings) 的邊界表現。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
import bisect

readings = [12.5, 3.8, 45.2, 19.0, 28.4]
readings.sort()

pos_min = bisect.bisect(readings, 0.0)
pos_max = bisect.bisect(readings, 100.0)

print("排序後讀數：", readings)
print(f"0.0 插入位置: {pos_min} (極左端邊界 0)")
print(f"100.0 插入位置: {pos_max} (極右端邊界 len = {len(readings)})")

### 12.9.2 `bisect.bisect_left(a, x)`：精準對應 Lower Bound（第一個 $\ge x$ 插入位置）

#### 1. 生活故事比喻：排隊買票時插在同伴的最前方
想像一隊依購票金額由少到多排隊的隊列：`[100, 200, 200, 200, 300]`。
現在有一位新來的顧客也要買 200 元的票。如果他非常有禮貌、或者他是同伴中的發起人，他想插隊在所有買 200 元顧客的**最左邊第一個（前面）**。
這就是 **`bisect_left`** 的物理意義！
英文中的 `left` 代表：**「若串列中已存在相同數值，新元素一律插入在所有現存相同數值的『左側（前方）』」**！

#### 2. 底層運作機制：無縫對接 Lower Bound
如果你回頭看 12.8.2 節我們手刻的 Lower Bound 代碼，你會驚喜地發現：
**`bisect.bisect_left(a, x)` 在數學上與 Lower Bound 100% 完全等價！**
其回傳的索引 `idx` 定義為：
- 滿足 $a[i] \ge x$ 的**第一個最小索引**！
- 如果 $x$ 存在於串列中：`idx` 必定是 $x$ **第一次（最左側）出現的下標**！
- 如果 $x$ 不在串列中：`idx` 是**第一個大於 $x$ 的元素下標**（也就是若將 $x$ 插入該處，串列仍維持升序的合法插入點）。

#### 3. 初學者常見陷阱：忘記透過 `idx < len(a) and a[idx] == x` 驗證存在性
許多初學者呼叫了 `idx = bisect.bisect_left(a, x)`，就以為 `a[idx]` 一定等於 `x`！
請再次銘記：如果 `x` 根本不在串列中，`bisect_left` 依然會回傳一個插入位置！
要確認 `x` 是否真的存在，必須加上防護雙條件：
```python
idx = bisect.bisect_left(a, x)
if idx < len(a) and a[idx] == x:
    # 這才代表真的存在！
```

#### 4. APCS 實戰視野
`bisect_left` 是 APCS 出現頻率最高的神級函數！無論是二分查找某數是否存在、找第一個達標的門檻、或是統計小於某數的元素個數（剛好就是 `idx` 個！），一行就能以 $O(\log N)$ 完美收工！

In [ ]:
# 範例 12.9.2：bisect.bisect_left() 實戰演練
import bisect

# 包含重複元素的已排序串列
data = [10, 20, 30, 30, 30, 30, 40, 50]
print("已排序資料：", data)
print("索引標註：", list(range(len(data))))

# 案例 1：在重複元素中鎖定最左側下標
idx_30 = bisect.bisect_left(data, 30)
print(f"\nbisect_left(data, 30) 回傳索引: {idx_30}")
print(f"驗證：data[{idx_30}] = {data[idx_30]}，正是第一個 30 出現的位置！")

# 案例 2：搜尋不存在數值，獲得維持升序之插入點
idx_25 = bisect.bisect_left(data, 25)
print(f"\nbisect_left(data, 25) 回傳索引: {idx_25}")
print(f"若將 25 插入在索引 {idx_25}，新串列依然維持嚴格升序！")

# 案例 3：極速統計「嚴格小於 30」的元素總共有幾個？
# 因為 idx_30 左邊的所有元素通通小於 30，所以個數剛好等於 idx_30！
less_than_30_count = idx_30
print(f"\n嚴格小於 30 的元素個數: {less_than_30_count} 個 (子清單: {data[:idx_30]})")

In [ ]:
# 填空題 12.9.2：使用 bisect_left 安全判定元素存在
# 任務：利用 bisect_left 判斷會員 ID 是否存在於已排序名冊中。
import bisect

members = [101, 205, 308, 412, 550]
query_id = 308

# 請呼叫 bisect_left 取得下界索引
idx = bisect.___(members, query_id)

# 請填寫雙重驗證：索引小於長度且數值等於 query_id
is_found = (idx < len(members)) and (members[idx] ___ query_id)

print(f"會員 {query_id} 是否存在：", is_found)

In [ ]:
# ==========================================
# [4] Code 練習題 12.9.2
# 任務說明：
# 給定已排序數列 grades = [45, 60, 60, 60, 75, 88, 92]。
# 請使用 bisect.bisect_left：
# 1. 找出第一個達到及格線 60 分的同學索引位置。
# 2. 計算「不及格（嚴格小於 60 分）」的同學一共有幾位。
#
# 【公開測試資料 1】
# grades = [45, 60, 60, 60, 75, 88, 92]
# 預期輸出：
# 第一個及格索引： 1
# 不及格人數： 1
#
# 【公開測試資料 2】
# grades = [60, 70, 80]
# 預期輸出：
# 第一個及格索引： 0
# 不及格人數： 0
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
import bisect

grades = [45, 60, 60, 60, 75, 88, 92]
idx_pass = bisect.bisect_left(grades, 60)
failed_count = idx_pass

print("第一個及格索引：", idx_pass)
print("不及格人數：", failed_count)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.9.2
# 任務說明：
# 某捷運公車刷卡系統維護了一串扣款門檻：zones = [15, 25, 35, 45, 60]
# 每次乘客里程數累積距離 dist，計費規則為「採用第一個大於或等於 dist 的門檻金額」。
# 若乘客搭乘距離超過 60，則一律收取最高上限 60 元。
# 請使用 bisect_left 撰寫計費程式，測試 dist = 28 與 dist = 70 的計費結果。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
import bisect

zones = [15, 25, 35, 45, 60]

def get_fare(dist):
    idx = bisect.bisect_left(zones, dist)
    if idx >= len(zones):
        return zones[-1]  # 超出上限收取最高額
    return zones[idx]

print("距離 28 公里車資：", get_fare(28), "元 (對應 35 元門檻)")
print("距離 70 公里車資：", get_fare(70), "元 (封頂 60 元)")

### 12.9.3 `bisect.bisect_right(a, x)`：精準對應 Upper Bound（第一個 $> x$ 插入位置）

#### 1. 生活故事比喻：排隊買票時排在同伴的最後方
依然是剛才排隊買票的比喻：`[100, 200, 200, 200, 300]`。
現在又來了一位要買 200 元票的新顧客。如果他非常客氣，說：「前面已經有三位朋友在買 200 元的票了，我不跟他們爭，我排在所有買 200 元朋友的**最右邊最後面**！」
這就是 **`bisect_right`**（在 Python 中也簡寫為 `bisect.bisect`）的物理行為！
它會把新元素安排在所有現存相同數值的**右側（後方）**！

#### 2. 底層運作機制：無縫對接 Upper Bound
`bisect.bisect_right(a, x)` 在數學上**100% 等價於我們在 12.8.3 節手刻的 Upper Bound**！
其回傳的索引 `idx` 定義為：
- 滿足 $a[i] > x$ 的**第一個嚴格大於目標的最小索引**！
- 如果要找「串列中最後一個等於 $x$ 的元素在哪裡」，答案剛好就是 **`idx - 1`**！
- 更棒的是：所有小於等於 $x$ 的元素，剛好通通落在索引 `0` 到 `idx - 1` 的範圍內，因此**「小於或等於 $x$ 的元素個數」剛好就等於 `idx`！**

#### 3. 初學者常見陷阱：`bisect` 與 `bisect_right` 的別名關係
請注意：在 Python 的 `bisect` 模組中，`bisect.bisect` 其實是 `bisect.bisect_right` 的**簡寫別名（Alias）**！
```python
bisect.bisect(a, x) == bisect.bisect_right(a, x)  # 兩者完全等價！
```
但在專業競技程式設計中，**強烈建議顯式寫出 `bisect_left` 或 `bisect_right`**！這能讓讀代碼的人一眼看出你到底是在尋找左邊界（$\ge$）還是右邊界（$>$），絕不會產生任何認知混淆。

#### 4. APCS 實戰視野
`bisect_left`（下界）與 `bisect_right`（上界）是成雙成對的黃金搭檔。下界鎖定起點，上界鎖定終點，兩者夾擊，構成了區間演算法的完美閉環！

In [ ]:
# 範例 12.9.3：bisect_left vs bisect_right 雙向對照實測
import bisect

data = [10, 20, 20, 20, 30, 40]
target = 20

print("資料串列：", data)
print("索引標註：", list(range(len(data))))

# 呼叫 bisect_left（下界，第一個 >= 20）
idx_left = bisect.bisect_left(data, target)

# 呼叫 bisect_right（上界，第一個 > 20）
idx_right = bisect.bisect_right(data, target)

print(f"\n針對數值 {target} 的雙邊界透視：")
print(f"  bisect_left  索引 = {idx_left}  (數值: {data[idx_left]}) -> 第一個 20 的位置")
print(f"  bisect_right 索引 = {idx_right} (數值: {data[idx_right]}) -> 第一個嚴格大於 20 的位置")
print(f"  最後一個 20 的位置 = idx_right - 1 = {idx_right - 1} (數值: {data[idx_right - 1]})")

# 統計小於等於 20 的元素總個數：
print(f"  小於等於 {target} 的元素個數剛好等於 idx_right: {idx_right} 個 (即 {data[:idx_right]})")

In [ ]:
# 填空題 12.9.3：定位最後一個出現位置
# 任務：利用 bisect_right 找出已排序數列中最後一個數值 50 的索引。
import bisect

scores = [20, 50, 50, 50, 50, 80, 95]
target = 50

# 取得第一個大於 50 的上界位置
ub = bisect.___(scores, target)

# 最後一個 50 的位置為 ub 減 1
last_pos = ub - ___

print(f"最後一個 {target} 位於索引：", last_pos)
print(f"驗證：scores[{last_pos}] = {scores[last_pos]}")

In [ ]:
# ==========================================
# [4] Code 練習題 12.9.3
# 任務說明：
# 給定已排序串列 arr = [5, 10, 15, 15, 15, 20, 25]。
# 請撰寫程式：
# 1. 呼叫 bisect_right 查詢數值 15 的上界索引。
# 2. 印出小於等於 15 的元素總個數（即該上界索引值）。
# 3. 輸出包含所有小於等於 15 的子串列。
#
# 【公開測試資料 1】
# arr = [5, 10, 15, 15, 15, 20, 25]
# 預期輸出：
# 15 的上界索引： 5
# <= 15 元素個數： 5
# 符合子串列： [5, 10, 15, 15, 15]
#
# 【公開測試資料 2】
# arr = [1, 2, 3]
# 預期輸出：
# 15 的上界索引： 3
# <= 15 元素個數： 3
# 符合子串列： [1, 2, 3]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
import bisect

arr = [5, 10, 15, 15, 15, 20, 25]
ub = bisect.bisect_right(arr, 15)

print("15 的上界索引：", ub)
print("<= 15 元素個數：", ub)
print("符合子串列：", arr[:ub])

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.9.3
# 任務說明：
# 某遊戲活動中，擊殺怪物的經驗值記錄已排序：exp_records = [10, 20, 20, 35, 50, 50, 50, 80]
# 請使用 bisect_left 與 bisect_right，
# 寫出一段程式，一次性印出數值 50 在串列中出現的「起始索引」與「結束索引」，
# 格式如：數值 50 的區間為 [start, end]。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
import bisect

exp_records = [10, 20, 20, 35, 50, 50, 50, 80]
target = 50

start_idx = bisect.bisect_left(exp_records, target)
end_idx = bisect.bisect_right(exp_records, target) - 1

print(f"數值 {target} 的區間為 [{start_idx}, {end_idx}]")
print(f"驗證：exp_records[{start_idx}] = {exp_records[start_idx]} ~ exp_records[{end_idx}] = {exp_records[end_idx]}")

### 12.9.4 數值區間計數神技：`bisect_right - bisect_left` 秒求元素個數與範圍 $[L, R]$ 統計

#### 1. 生活故事比喻：量尺上的刻度差
在木工測量中，如果你想知道木板上一段特定花紋的長度，你不需要拿放大鏡一毫米一毫米去數。
你只需拿出捲尺：
- 記下花紋起點的刻度 $A$（`bisect_left`）。
- 記下花紋終點後方的下一個刻度 $B$（`bisect_right`）。
拿終點減去起點：$B - A$，這段花紋的長度與數量立刻精確求出！
在 Python 競賽中，當面對百萬筆數據，如果題目問你：「特定分數考了幾個人？」、「介於 70 分到 85 分之間的考生有幾個人？」
只要兩行 `bisect` 相減，**連一個迴圈都不用寫，瞬間以 $O(\log N)$ 秒殺全場！**

#### 2. 底層運作機制：兩大必備區間計數公式
給定一個已排序串列 `arr`：
- **公式 1：單一數值 $X$ 的出現總次數**
  $$\text{Count}(X) = \text{bisect\_right}(arr, X) - \text{bisect\_left}(arr, X)$$
- **公式 2：數值範圍在閉區間 $[Low, High]$ 內的元素總個數**
  - 第一步：找出第一個小於等於 $High$ 的上界邊界：$R = \text{bisect\_right}(arr, High)$。
  - 第二步：找出第一個大於等於 $Low$ 的下界邊界：$L = \text{bisect\_left}(arr, Low)$。
  - 區間總個數即為：
    $$\text{Count}(Low \le x \le High) = R - L$$

#### 3. 初學者常見陷阱：閉區間與開區間的函數選用混淆
初學同學在算範圍 $[Low, High]$ 時常搞混：
- 下界 $Low$：因為要「包含 $Low$」，所以要找「第一個 $\ge Low$ 的人」，**必用 `bisect_left`**！
- 上界 $High$：因為要「包含 $High$」，所以要找「第一個嚴格 $> High$ 的人」作為右邊界哨兵，**必用 `bisect_right`**！
記住口訣：**「左閉用 left，右閉用 right」**！

#### 4. APCS 實戰視野
APCS 歷屆實作考題中多次出現「給定多組區間查詢 $[L, R]$，統計符合條件的觀測點數量」。若用純迴圈，每次查詢花費 $O(N)$，總時間 $O(Q \times N)$ 必吃超時（TLE）。改用此雙劍合璧神技，總時間瞬間降至 $O(Q \log N)$，0.05 秒斬獲滿分！

In [ ]:
# 範例 12.9.4：雙界相減計數神技演示
import bisect

# 全體學生的英文段考成績已排序
scores = [45, 52, 60, 60, 60, 72, 78, 85, 85, 90, 95, 100]
print("全體成績名冊：", scores)

# 任務 1：極速計算剛好考 60 分的人數
count_60 = bisect.bisect_right(scores, 60) - bisect.bisect_left(scores, 60)
print(f"\n1. 考 60 分的人數: {count_60} 人 (完全免寫迴圈！)")

# 任務 2：極速統計成績落在 [70, 89] 分之間（乙等）的總人數
# 左閉右閉區間 [70, 89] -> 左邊用 left(70)，右邊用 right(89)
L_idx = bisect.bisect_left(scores, 70)    # 第一個 >= 70
R_idx = bisect.bisect_right(scores, 89)   # 第一個 > 89
count_70_89 = R_idx - L_idx

print(f"2. 成績介於 [70, 89] 分之間的人數: {count_70_89} 人")
print(f"   符合條件的名單切片: {scores[L_idx:R_idx]}")

In [ ]:
# 填空題 12.9.4：區間查詢個數填空
# 任務：計算數列中數值落在 [20, 40] 之間的元素個數。
import bisect

numbers = [5, 12, 20, 25, 30, 40, 45, 60]

# 左邊界 20 包含：使用 bisect_left
l = bisect.___(numbers, 20)
# 右邊界 40 包含：使用 bisect_right
r = bisect.___(numbers, 40)

count = r - l
print("落在 [20, 40] 之間的元素個數為：", count)

In [ ]:
# ==========================================
# [4] Code 練習題 12.9.4
# 任務說明：
# 給定已排序的身高清單 heights 與多組查詢區間 queries（每組為 (min_h, max_h)）。
# 請使用 bisect_right - bisect_left 計算每組區間內的人數清單並印出。
#
# 【公開測試資料 1】
# heights = [150, 155, 160, 165, 170, 175, 180, 185]
# queries = [(160, 170), (150, 150), (190, 200)]
# 預期輸出：
# 各區間人數： [3, 1, 0]
#
# 【公開測試資料 2】
# heights = [100, 200]
# queries = [(50, 150), (100, 200)]
# 預期輸出：
# 各區間人數： [1, 2]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
import bisect

heights = [150, 155, 160, 165, 170, 175, 180, 185]
queries = [(160, 170), (150, 150), (190, 200)]

def count_in_range(arr, low, high):
    l = bisect.bisect_left(arr, low)
    r = bisect.bisect_right(arr, high)
    return r - l

res = [count_in_range(heights, q[0], q[1]) for q in queries]
print("各區間人數：", res)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.9.4
# 任務說明：
# 某量販店促銷，凡是購買金額落在 [500, 1500] 之間的訂單可獲得折價券。
# 今日已排序訂單金額：orders = [120, 450, 500, 890, 1200, 1500, 1500, 2300, 3100]
# 請使用 bisect 模組計算共有幾筆訂單可獲得折價券，
# 並計算這些符合資格訂單的「總消費金額（使用切片加總 sum）」。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
import bisect

orders = [120, 450, 500, 890, 1200, 1500, 1500, 2300, 3100]
l = bisect.bisect_left(orders, 500)
r = bisect.bisect_right(orders, 1500)

qualified_count = r - l
qualified_sum = sum(orders[l:r])

print(f"符合折價券資格的訂單共: {qualified_count} 筆")
print(f"符合訂單金額總計: {qualified_sum} 元")

### 12.9.5 數值級距查表神器：利用 bisect 快速實作等第成績評定（免寫 10 個 if-elif 分支）

#### 1. 生活故事比喻：郵件秤重計費的自動落槽箱
在郵局寄包裹時，櫃台後方有一座設計精巧的分類落槽箱：
- 不超過 50 克：掉進「平信槽（8 元）」。
- 不超過 100 克：掉進「小包槽（15 元）」。
- 不超過 250 克：掉進「大包槽（25 元）」。
郵務人員把包裹隨手往上一扔，包裹順著重量自動滑落到正確的格子裡。
如果用傳統程式碼來寫這段邏輯，初學者會寫出一長串又臭又長、縮排深不見底的 `if-elif-elif-elif-else` 分支。
但如果我們用 `bisect`，我們只需要準備一組**「門檻斷點清單（Breakpoints）」**與一組**「對應等第清單（Grades）」**，一行 `bisect` 就能像那個自動落槽箱一樣，瞬間將任何分數分類到正確的等第中！

#### 2. 底層運作機制：斷點陣列與索引映射的數學魔法
讓我們看經典的學校成績等第換算：
- 60 分以下：`'F'`
- 60 到 69 分：`'D'`
- 70 到 79 分：`'C'`
- 80 到 89 分：`'B'`
- 90 分以上：`'A'`

我們設計一組斷點清單：`breakpoints = [60, 70, 80, 90]`。  
對應的等第清單：`grades = ['F', 'D', 'C', 'B', 'A']`（剛好比斷點多 1 個元素！）。
現在測試：
- 考 55 分：`bisect(breakpoints, 55)` 找到插入點索引 `0` ➔ `grades[0]` 為 `'F'`！
- 考 60 分：`bisect(breakpoints, 60)` 因為 60 已達標，`bisect_right` 會跳到索引 `1` ➔ `grades[1]` 為 `'D'`！
- 考 85 分：找到索引 `3` ➔ `grades[3]` 為 `'B'`！
- 考 99 分：找到索引 `4` ➔ `grades[4]` 為 `'A'`！
完全不需要撰寫任何一個 `if` 或 `elif`！

#### 3. 初學者常見陷阱：邊界包含（及格是 60 分還是 61 分）
請注意：
若規定「剛好 60 分為 D 等第」，呼叫 `bisect_right` 時，60 會落入索引 1（D），完全吻合！
但如果規定「剛好 60 分還是 F」，那就要換成 `bisect_left`。因此務必釐清題目邊界是否包含端點。

#### 4. APCS 實戰視野
查表法（Lookup Table）搭配二分級距查詢，是高階軟體架構中「資料驅動程式設計（Data-driven Design）」的經典典範。當未來門檻規則隨時變更時，你只需修改 `breakpoints` 陣列，業務代碼一行都不用動！

In [ ]:
# 範例 12.9.5：bisect 實現極速等第查表神器
import bisect

# 定義門檻斷點清單與對應等第標籤
breakpoints = [60, 70, 80, 90]
grades = ["F", "D", "C", "B", "A"]

def score_to_grade(score):
    # 使用 bisect_right（或預設 bisect）
    idx = bisect.bisect_right(breakpoints, score)
    return grades[idx]

# 測試各種不同分數
test_scores = [35, 60, 69, 70, 82, 89, 90, 100]

print("成績與等第轉換對照表（免寫 if-elif 分支！）：")
for s in test_scores:
    print(f"  分數: {s:3d} 分 -> 評定等第: {score_to_grade(s)}")

In [ ]:
# 填空題 12.9.5：所得稅率級距查表
# 任務：年所得（萬元）對應稅率級距：
# 50 萬以下：5%，超過 50 萬到 120 萬：12%，超過 120 萬：20%。
import bisect

tax_brackets = [50, 120]
tax_rates = ["5%", "12%", "20%"]

income = 80  # 年所得 80 萬元
# 請呼叫 bisect.bisect 計算級距索引
bracket_idx = bisect.___(tax_brackets, income)

print(f"年所得 {income} 萬元，適用稅率為：", tax_rates[bracket_idx])

In [ ]:
# ==========================================
# [4] Code 練習題 12.9.5
# 任務說明：
# 某遊戲依積分排定段位：
# 積分 thresholds = [1000, 2000, 3000]
# 段位 ranks = ["青銅", "白銀", "黃金", "鑽石"]
# 請撰寫函數 get_rank(points)，利用 bisect 查表回傳對應段位名稱。
#
# 【公開測試資料 1】
# points = 2500
# 預期輸出：
# 積分 2500 段位： 黃金
#
# 【公開測試資料 2】
# points = 800
# 預期輸出：
# 積分 800 段位： 青銅
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
import bisect

thresholds = [1000, 2000, 3000]
ranks = ["青銅", "白銀", "黃金", "鑽石"]

def get_rank(points):
    idx = bisect.bisect_right(thresholds, points)
    return ranks[idx]

print("積分 2500 段位：", get_rank(2500))
print("積分 800 段位：", get_rank(800))

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.9.5
# 任務說明：
# 某購物網站依消費金額打折：
# 滿 1000 打 9 折 (0.90)，滿 3000 打 8 折 (0.80)，滿 5000 打 7 折 (0.70)，未滿 1000 不打折 (1.00)。
# 請建立斷點與折扣率清單，設計一個計算結帳金額的函數 calc_final_bill(amount)，
# 並印出消費 2500 元與 6000 元的實付金額。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
import bisect

discount_brackets = [1000, 3000, 5000]
discount_rates = [1.00, 0.90, 0.80, 0.70]

def calc_final_bill(amount):
    idx = bisect.bisect_right(discount_brackets, amount)
    rate = discount_rates[idx]
    return int(amount * rate)

print("消費 2500 元實付：", calc_final_bill(2500), "元 (打 9 折)")
print("消費 6000 元實付：", calc_final_bill(6000), "元 (打 7 折)")

### 12.9.6 維護動態有序序列：`bisect.insort()` 高效插入與單調答案值域二分搜尋初探

#### 1. 生活故事比喻：隨時插隊的撲克牌理牌手
當你在玩大老二或橋牌時，發牌員每發一張新牌到你手裡，你不會把手上的所有牌全部扔到桌上重新洗牌、重新由小到大排一遍！
你會用眼睛掃一下手上原本已經排好的牌，找到那張新牌該放的空隙，輕輕把牌插進去，手上的牌依然保持整整齊齊！
在 Python 程式中，如果你每次加一個新數字都呼叫一次 `.sort()`，每次都要花費 $O(N \log N)$ 的重排代價；如果加 1,000 次新數字，電腦會跑得痛苦不堪。
但如果使用 **`bisect.insort()`**：
它會先用二分搜尋在 $O(\log N)$ 內秒殺定位出合適的空隙，然後直接插入，這就是動態維護有序序列的終極絕招！

#### 2. 底層運作機制：`insort_left` 與 `insort_right` 的插入操作
- `bisect.insort_right(a, x)`（簡寫為 `bisect.insort(a, x)`）：  
  先以二分搜尋找出插入位置，再執行串列原地插入 `a.insert(idx, x)`。
- `bisect.insort_left(a, x)`：  
  若有相同數值，插入在現存相同數值的左側。
無論何時調用 `insort()`，串列隨時隨地保證處於**嚴格升序排序狀態**！

#### 3. APCS 頂峰預告：單調答案值域二分搜尋（Binary Search on Answer）
在 APCS 實作第四級乃至各類全國資訊奧林匹亞決賽中，二分搜尋最高級的應用，根本不是在「串列」裡找數字，而是在**「抽象的答案可能範圍（值域）」**上進行二分！
例如：
- 「已知有 $K$ 台抽水機，請問最少需要花費『幾分鐘』能把水庫排乾？」
因為答案具有單調性（如果 50 分鐘能排乾，那麼 60 分鐘一定也能排乾！），我們可以在 `[0, 1000000]` 的時間值域上進行二分搜尋！
掌握了今天 `bisect` 與邊界收斂的精髓，你已經完全具備了邁向這個演算法最高峰的一切底層功力！

#### 4. APCS 實戰視野
從靜態的 `sort()` 排序，到動態的 `bisect.insort()` 維護，再到答案值域的二分逼近，整個第十二章為你構建了一座從語法通往頂尖演算法的巍峨長橋！

In [ ]:
# 範例 12.9.6：bisect.insort() 動態維護有序序列
import bisect

# 初始時已排序的玩家即時排行榜
leaderboard = [100, 250, 400, 650, 800]
print("初始排行榜：", leaderboard)

# 模擬新玩家陸續產生新分數
new_scores = [350, 100, 950, 500]

for score in new_scores:
    # 使用 bisect.insort() 一步到位：二分定位 + 自動插入
    bisect.insort(leaderboard, score)
    print(f"  新分數 {score:3d} 插入後，排行榜維持有序: {leaderboard}")

# 驗證：隨時隨地，榜單第一位必為最小值，末位必為最大值
print("\n最終動態維護的有序串列：", leaderboard)
print(f"全服最低分: {leaderboard[0]}, 全服最高分: {leaderboard[-1]}")

In [ ]:
# 填空題 12.9.6：動態插入新測量值
# 任務：使用 bisect.insort 將溫度讀數插入已排序串列中。
import bisect

live_temps = [18.5, 20.2, 22.0, 25.5]
new_reading = 21.4

# 請呼叫 bisect.insort 進行動態有序插入
bisect.___(live_temps, new_reading)

print("動態插入後的溫度監控表：", live_temps)

In [ ]:
# ==========================================
# [4] Code 練習題 12.9.6
# 任務說明：
# 某即時待辦事項系統以緊急度（整數數值，數值越小越緊急）排序。
# 初始待辦清單：tasks = [1, 3, 5, 8]
# 請依序使用 bisect.insort 將新任務 [4, 2, 7] 插入清單中，
# 並輸出最終維持有序的任務緊急度清單。
#
# 【公開測試資料 1】
# 初始 tasks = [1, 3, 5, 8]，插入新任務 [4, 2, 7]
# 預期輸出：
# 最終任務清單： [1, 2, 3, 4, 5, 7, 8]
#
# 【公開測試資料 2】
# 初始 tasks = [10]，插入新任務 [5]
# 預期輸出：
# 最終任務清單： [5, 10]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
import bisect

tasks = [1, 3, 5, 8]
new_tasks = [4, 2, 7]

for t in new_tasks:
    bisect.insort(tasks, t)

print("最終任務清單：", tasks)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.9.6
# 任務說明：
# 請設計一個微型的「動態中位數（Running Median）監控器」原型：
# 初始串列為空 stream = []。
# 依序讀入 5 個串流數字：[40, 10, 30, 50, 20]。
# 每讀入一個數字，使用 bisect.insort 插入，並印出當前串列與其正中間的中位數數值（len // 2 處的元素）。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
import bisect

stream = []
incoming = [40, 10, 30, 50, 20]

print("動態串流數據監控：")
for num in incoming:
    bisect.insort(stream, num)
    mid_val = stream[len(stream) // 2]
    print(f"  流入 {num:2d} -> 當前數列: {stream}, 中位數: {mid_val}")

### 學習總結與通關回顧

恭喜你順利通關 **12.9 內建二分搜尋模組：bisect 與數值區間查詢應用**！
更要熱烈慶祝你**全數征服第十二章「排序、搜尋演算法與相關模組」全體 9 大小節！**

在本單元中，你達成了標準函式庫與演算法實戰的終極合體：
- **`bisect` 官方模組的 C 語言極速優勢**：
  - 零外部依賴、純 C 語言加速，APCS 考場必備合法利器。
- **雙邊界與手刻概念的完美映射**：
  - `bisect.bisect_left(a, x)` 完美對應 Lower Bound（第一個 $\ge x$）。
  - `bisect.bisect_right(a, x)` 完美對應 Upper Bound（第一個 $> x$）。
- **區間計數神技**：
  - 單元個數：`bisect_right(x) - bisect_left(x)`。
  - 閉區間 $[L, R]$ 個數：`bisect_right(R) - bisect_left(L)`。
- **等第級距查表法**：
  - 斷點陣列搭配 `bisect_right`，1 行取代 10 個 `if-elif` 繁瑣分支。
- **動態有序維護**：
  - `bisect.insort()` 隨時插入新元素，終身保持單調有序。

---
**下一章預告**：寫程式最怕什麼？上傳系統噴出令人心碎的 `WA`、`TLE`、`RE`！下一章 **第十三章 程式除錯（Debug）與異常處理** 將帶你化身專業的代碼神醫，剖析所有評測結果與除錯心法，讓你不再害怕任何 Bug！